# Notebook 05 — Econometric Analysis
**Project:** RBI Communication, Financial Stability and Macroeconomic Transmission: Evidence from Sentiment Analysis of MPC Minutes (2010-2025)

### Pipeline
1. Descriptive Statistics
2. ADF Unit Root Test
3. Correlation Analysis
4. OLS Regression
5. Granger Causality Test
6. VAR Model
7. Event Study Analysis

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.stats.stattools import durbin_watson
from scipy import stats

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PROC  = os.path.join(PROJECT_ROOT, 'data', 'processed')
OUTPUT_TAB = os.path.join(PROJECT_ROOT, 'output', 'tables')
OUTPUT_RES = os.path.join(PROJECT_ROOT, 'output', 'results')
os.makedirs(OUTPUT_TAB, exist_ok=True)
os.makedirs(OUTPUT_RES, exist_ok=True)

master_path = os.path.join(DATA_PROC, 'master_dataset.csv')
if os.path.exists(master_path):
    df = pd.read_csv(master_path, parse_dates=['Date'])
else:
    for fname in ['combined_sentiment_lm.csv','combined_sentiment_aligned.csv']:
        p = os.path.join(DATA_PROC, fname)
        if os.path.exists(p):
            df = pd.read_csv(p)
            if 'Period' in df.columns:
                df.rename(columns={'Period':'Date'}, inplace=True)
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
            break

df = df.sort_values('Date').reset_index(drop=True)
print("Dataset: %d rows x %d cols" % (df.shape[0], df.shape[1]))
print(df.columns.tolist())

Dataset: 31 rows x 31 cols
['Date', 'MPC_Sentiment', 'MPC_Polarity', 'MPC_Positive', 'MPC_Negative', 'MPC_TotalWords', 'FSR_Sentiment', 'FSR_Polarity', 'FSR_Positive', 'FSR_Negative', 'FSR_TotalWords', 'FSSI_raw', 'FSSI', 'FSSI_smoothed', 'MPSI', 'MPSI_smoothed', 'Repo_Rate', 'Repo_Change', 'Stance', 'Event', 'Repo_Rate_macro', 'CPI_Inflation', 'WPI_Inflation', 'GDP_Growth', 'Bank_Credit_Growth', 'Nifty50', 'BankNifty', 'USD_INR', 'Nifty50_Return', 'BankNifty_Return', 'USD_INR_Return']


## Step 1 — Descriptive Statistics

In [2]:
ALL_VARS = ['MPC_Sentiment','FSR_Sentiment','MPSI','FSSI',
            'Repo_Rate','CPI_Inflation','GDP_Growth',
            'Bank_Credit_Growth','Nifty50','USD_INR']
KEY_VARS = [v for v in ALL_VARS if v in df.columns]
print("Available variables:", KEY_VARS)

desc = df[KEY_VARS].describe().round(4)
print("\n" + "="*70)
print("TABLE 1 — DESCRIPTIVE STATISTICS")
print("="*70)
print(desc.to_string())
desc.to_csv(os.path.join(OUTPUT_TAB, 'table1_descriptive_stats.csv'))
print("\nSaved: table1_descriptive_stats.csv")

Available variables: ['MPC_Sentiment', 'FSR_Sentiment', 'MPSI', 'FSSI', 'Repo_Rate', 'CPI_Inflation', 'GDP_Growth', 'Bank_Credit_Growth', 'Nifty50', 'USD_INR']

TABLE 1 — DESCRIPTIVE STATISTICS
       MPC_Sentiment  FSR_Sentiment      MPSI      FSSI  Repo_Rate  CPI_Inflation  GDP_Growth  Bank_Credit_Growth     Nifty50  USD_INR
count        31.0000        31.0000   31.0000   31.0000    31.0000        31.0000     31.0000             31.0000     31.0000  31.0000
mean         -0.0022        -0.0055   57.7883   38.1667     6.3403         6.3774      5.7226             13.2484  11640.0852  67.5458
std           0.0038         0.0014   25.1462   29.1389     1.2562         2.3918      5.9822              4.6437   6064.8133  11.7829
min          -0.0110        -0.0085    0.0000    0.0000     4.0000         2.2000    -24.4000              5.5000   5025.7200  44.9500
25%          -0.0044        -0.0063   43.4127   15.1953     5.8750         4.8500      5.4000              9.3500   6617.6700  61.6

## Step 2 — ADF Unit Root Test

In [3]:
def adf_test(series, name):
    s = series.dropna()
    if len(s) < 8:
        return {'Variable':name,'ADF_Stat':np.nan,'p_value':np.nan,'Stationary':'N/A'}
    r = adfuller(s, autolag='AIC')
    p = r[1]
    return {
        'Variable'   : name,
        'ADF_Stat'   : round(r[0], 4),
        'p_value'    : round(p, 4),
        'Lags'       : r[2],
        'Crit_1pct'  : round(r[4]['1%'], 4),
        'Crit_5pct'  : round(r[4]['5%'], 4),
        'Stationary' : 'Yes' if p < 0.05 else 'No',
        'Decision'   : 'Stationary I(0)' if p < 0.05 else 'Non-Stationary — difference'
    }

adf_list = [adf_test(df[v], v) for v in KEY_VARS]
adf_df = pd.DataFrame(adf_list)
print("="*70)
print("TABLE 2 — ADF UNIT ROOT TEST")
print("="*70)
print(adf_df.to_string(index=False))
adf_df.to_csv(os.path.join(OUTPUT_TAB, 'table2_adf_unit_root.csv'), index=False)

non_stat = adf_df[adf_df['Stationary']=='No']['Variable'].tolist()
print("\nNon-stationary vars (will be differenced):", non_stat)
for v in non_stat:
    if v in df.columns:
        df['d_'+v] = df[v].diff()
        print("  Created: d_%s" % v)

TABLE 2 — ADF UNIT ROOT TEST
          Variable  ADF_Stat  p_value  Lags  Crit_1pct  Crit_5pct Stationary                    Decision
     MPC_Sentiment   -5.2379   0.0000     0    -3.6699    -2.9641        Yes             Stationary I(0)
     FSR_Sentiment   -3.3070   0.0146     0    -3.6699    -2.9641        Yes             Stationary I(0)
              MPSI   -5.2379   0.0000     0    -3.6699    -2.9641        Yes             Stationary I(0)
              FSSI   -3.3070   0.0146     0    -3.6699    -2.9641        Yes             Stationary I(0)
         Repo_Rate   -2.0028   0.2854     1    -3.6791    -2.9679         No Non-Stationary — difference
     CPI_Inflation   -2.7428   0.0669     5    -3.7239    -2.9865         No Non-Stationary — difference
        GDP_Growth   -4.3486   0.0004     0    -3.6699    -2.9641        Yes             Stationary I(0)
Bank_Credit_Growth   -3.1515   0.0230     1    -3.6791    -2.9679        Yes             Stationary I(0)
           Nifty50    1.38

## Step 3 — Correlation Analysis

In [4]:
corr = df[KEY_VARS].corr(method='pearson').round(4)
print("="*70)
print("TABLE 3 — PEARSON CORRELATION MATRIX")
print("="*70)
print(corr.to_string())
corr.to_csv(os.path.join(OUTPUT_TAB, 'table3_correlation_matrix.csv'))
print("\nSaved: table3_correlation_matrix.csv")

mpc_col = 'MPC_Sentiment' if 'MPC_Sentiment' in corr.columns else KEY_VARS[0]
print("\nCorrelations with %s (ranked by absolute value):" % mpc_col)
sub = corr[mpc_col].drop(mpc_col).sort_values(key=abs, ascending=False)
for var, val in sub.items():
    s = " ***" if abs(val)>0.5 else (" **" if abs(val)>0.3 else ("  *" if abs(val)>0.2 else ""))
    print("  %-28s: %7.4f%s" % (var, val, s))

TABLE 3 — PEARSON CORRELATION MATRIX
                    MPC_Sentiment  FSR_Sentiment    MPSI    FSSI  Repo_Rate  CPI_Inflation  GDP_Growth  Bank_Credit_Growth  Nifty50  USD_INR
MPC_Sentiment              1.0000         0.3079  1.0000 -0.3079    -0.2127         0.0134      0.3283              0.1434   0.2664   0.0710
FSR_Sentiment              0.3079         1.0000  0.3079 -1.0000    -0.2270        -0.4421      0.3493              0.0990   0.3958   0.1978
MPSI                       1.0000         0.3079  1.0000 -0.3079    -0.2127         0.0134      0.3283              0.1434   0.2664   0.0710
FSSI                      -0.3079        -1.0000 -0.3079  1.0000     0.2270         0.4421     -0.3493             -0.0990  -0.3958  -0.1978
Repo_Rate                 -0.2127        -0.2270 -0.2127  0.2270     1.0000         0.3766      0.3146              0.4523  -0.4357  -0.4847
CPI_Inflation              0.0134        -0.4421  0.0134  0.4421     0.3766         1.0000     -0.0002              0

## Step 4 — OLS Regression

In [5]:
mpc_var = 'MPC_Sentiment' if 'MPC_Sentiment' in df.columns else 'MPSI'
fsr_var = 'FSR_Sentiment' if 'FSR_Sentiment' in df.columns else 'FSSI'

def run_ols(y_var, x_vars):
    cols = [y_var] + x_vars
    data = df[[c for c in cols if c in df.columns]].dropna()
    avail_x = [c for c in x_vars if c in data.columns]
    if len(data) < 10 or not avail_x:
        return None, None
    y = data[y_var]
    X = sm.add_constant(data[avail_x])
    model = sm.OLS(y, X).fit()
    dw = durbin_watson(model.resid)
    out = {'Dep_Var': y_var, 'N': int(model.nobs),
           'R2': round(model.rsquared, 4),
           'Adj_R2': round(model.rsquared_adj, 4),
           'F': round(model.fvalue, 4),
           'F_p': round(model.f_pvalue, 4),
           'DW': round(dw, 4)}
    for x in avail_x:
        out[x+'_coef'] = round(model.params.get(x, np.nan), 4)
        out[x+'_pval'] = round(model.pvalues.get(x, np.nan), 4)
    return out, model

dep_vars = [v for v in ['CPI_Inflation','GDP_Growth','Bank_Credit_Growth',
                         'Nifty50','USD_INR','FSSI'] if v in df.columns]

ols_rows = []
print("="*70)
print("TABLE 4 — OLS REGRESSION RESULTS")
print("="*70)
for dv in dep_vars:
    regs = [v for v in [mpc_var, fsr_var] if v in df.columns and v != dv]
    res, mdl = run_ols(dv, regs)
    if res:
        ols_rows.append(res)
        sig = "***" if res['F_p']<0.01 else ("**" if res['F_p']<0.05 else ("*" if res['F_p']<0.10 else ""))
        print("\nDependent: %-25s N=%-3d R2=%.4f Adj-R2=%.4f F-p=%.4f %s" % (
              dv, res['N'], res['R2'], res['Adj_R2'], res['F_p'], sig))
        for v in regs:
            if v in df.columns:
                c = res.get(v+'_coef','N/A')
                p = res.get(v+'_pval','N/A')
                print("    %-28s coef=%-10s pval=%s" % (v, c, p))

pd.DataFrame(ols_rows).to_csv(os.path.join(OUTPUT_TAB,'table4_ols_regression.csv'), index=False)
print("\nSaved: table4_ols_regression.csv")

TABLE 4 — OLS REGRESSION RESULTS

Dependent: CPI_Inflation             N=31  R2=0.2202 Adj-R2=0.1644 F-p=0.0308 **
    MPC_Sentiment                coef=103.4147   pval=0.3544
    FSR_Sentiment                coef=-836.68    pval=0.0089

Dependent: GDP_Growth                N=31  R2=0.1758 Adj-R2=0.1169 F-p=0.0667 *
    MPC_Sentiment                coef=381.8258   pval=0.1872
    FSR_Sentiment                coef=1164.083   pval=0.1396

Dependent: Bank_Credit_Growth        N=31  R2=0.0239 Adj-R2=-0.0458 F-p=0.7128 
    MPC_Sentiment                coef=151.6783   pval=0.53
    FSR_Sentiment                coef=199.5375   pval=0.7599

Dependent: Nifty50                   N=31  R2=0.1797 Adj-R2=0.1211 F-p=0.0625 *
    MPC_Sentiment                coef=253417.2094 pval=0.3825
    FSR_Sentiment                coef=1491805.1219 pval=0.0642

Dependent: USD_INR                   N=31  R2=0.0392 Adj-R2=-0.0294 F-p=0.5709 
    MPC_Sentiment                coef=34.272     pval=0.9549
    FSR_Sen

## Step 5 — Granger Causality Test

In [6]:
def granger_test(cause, effect, max_lag=4):
    data = df[[cause, effect]].dropna()
    if len(data) < max_lag + 6:
        return {'Cause':cause,'Effect':effect,'Best_p':'N/A','Best_lag':'N/A','Sig':'N/A'}
    try:
        ml = min(max_lag, max(1, len(data)//5))
        res = grangercausalitytests(data[[effect, cause]], maxlag=ml, verbose=False)
        best_p, best_lag = 1.0, 1
        for lag, r in res.items():
            p = r[0]['ssr_ftest'][1]
            if p < best_p:
                best_p, best_lag = p, lag
        sig = '***' if best_p<0.01 else ('**' if best_p<0.05 else ('*' if best_p<0.10 else 'No'))
        return {'Cause':cause,'Effect':effect,
                'Best_Lag':best_lag,'Best_p':round(best_p,4),'Sig':sig}
    except Exception as e:
        return {'Cause':cause,'Effect':effect,'Best_p':'Error','Sig':'N/A','Note':str(e)[:60]}

targets = [v for v in ['CPI_Inflation','GDP_Growth','Nifty50',
                        'Bank_Credit_Growth','USD_INR',fsr_var]
           if v in df.columns and v != mpc_var]

gc_rows = []
print("="*70)
print("TABLE 5 — GRANGER CAUSALITY TEST")
print("H0: MPC Sentiment does NOT Granger-cause target variable")
print("="*70)
for t in targets:
    r = granger_test(mpc_var, t)
    gc_rows.append(r)
    print("  %-30s -> %-25s: p=%-8s lag=%-3s %s" % (
          mpc_var, t, r['Best_p'], r.get('Best_Lag','N/A'), r['Sig']))

pd.DataFrame(gc_rows).to_csv(os.path.join(OUTPUT_TAB,'table5_granger_causality.csv'), index=False)
print("\n*** p<0.01  ** p<0.05  * p<0.10")
print("Saved: table5_granger_causality.csv")

TABLE 5 — GRANGER CAUSALITY TEST
H0: MPC Sentiment does NOT Granger-cause target variable
  MPC_Sentiment                  -> CPI_Inflation            : p=0.1006   lag=3   No
  MPC_Sentiment                  -> GDP_Growth               : p=0.2993   lag=2   No
  MPC_Sentiment                  -> Nifty50                  : p=0.1516   lag=3   No
  MPC_Sentiment                  -> Bank_Credit_Growth       : p=0.211    lag=1   No
  MPC_Sentiment                  -> USD_INR                  : p=0.0111   lag=4   **
  MPC_Sentiment                  -> FSR_Sentiment            : p=0.0603   lag=1   *

*** p<0.01  ** p<0.05  * p<0.10
Saved: table5_granger_causality.csv


## Step 6 — VAR Model

In [7]:
var_candidates = [v for v in [mpc_var,'CPI_Inflation','GDP_Growth','Nifty50']
                   if v in df.columns and df[v].notna().sum() >= 15]
print("VAR variables:", var_candidates)

var_data = df[var_candidates].dropna()
print("Observations for VAR:", len(var_data))

if len(var_data) >= 12 and len(var_candidates) >= 2:
    try:
        model_var = VAR(var_data)
        lo = model_var.select_order(maxlags=min(4, len(var_data)//5))
        print("\nLag order criteria:")
        print(lo.summary())

        best_lag = max(1, lo.selected_orders.get('aic', 2))
        var_result = model_var.fit(best_lag)
        print("\nVAR(%d) Summary:" % best_lag)
        print(var_result.summary())

        with open(os.path.join(OUTPUT_TAB,'table6_var_model.txt'), 'w') as f:
            f.write(str(var_result.summary()))
        print("Saved: table6_var_model.txt")

        # Impulse Response Function
        irf = var_result.irf(periods=8)
        irf_rows = []
        for i, rv in enumerate(var_candidates):
            for j, iv in enumerate(var_candidates):
                for t in range(9):
                    irf_rows.append({'Impulse':iv,'Response':rv,
                                     'Period':t,'IRF':round(float(irf.irfs[t][i][j]),6)})
        pd.DataFrame(irf_rows).to_csv(
            os.path.join(OUTPUT_RES,'var_impulse_response.csv'), index=False)
        print("Saved: var_impulse_response.csv")
    except Exception as e:
        print("VAR error:", e)
else:
    print("Not enough data for VAR — need >=12 obs and >=2 variables.")

VAR variables: ['MPC_Sentiment', 'CPI_Inflation', 'GDP_Growth', 'Nifty50']
Observations for VAR: 31

Lag order criteria:
 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0       11.37       11.56   8.636e+04       11.42
1       7.438      8.398*       1729.       7.723
2       7.032       8.760       1259.       7.546
3      6.725*       9.221      1179.*      7.467*
4       6.884       10.15       2376.       7.854
-------------------------------------------------

VAR(3) Summary:
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 26, May, 2026
Time:                     16:16:18
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                    9.64079
Nobs:                     28.0000    HQIC:                   7.92305
Log likelihood:          -207.

## Step 7 — Event Study Analysis

In [8]:
KEY_EVENTS = {
    '2013-07':'Taper Tantrum',
    '2016-10':'MPC Framework Launch',
    '2016-12':'Demonetisation Shock',
    '2019-06':'Rate Cut Cycle Start',
    '2020-06':'COVID Emergency Cut',
    '2022-06':'Post-COVID Rate Hike',
    '2024-12':'Rate Cut Restart',
}
df['YM'] = df['Date'].dt.strftime('%Y-%m')

ev_rows = []
for ym, ename in KEY_EVENTS.items():
    idx_list = df[df['YM'] == ym].index.tolist()
    if not idx_list:
        continue
    idx = idx_list[0]
    row = {'Event': ename, 'Date': ym}
    row['MPC_Sentiment'] = round(df.loc[idx, mpc_var], 4) if mpc_var in df.columns else np.nan
    if 'Repo_Rate' in df.columns:
        row['Repo_Rate'] = df.loc[idx, 'Repo_Rate']
        if idx > 0:
            row['Rate_Change'] = df.loc[idx,'Repo_Rate'] - df.loc[idx-1,'Repo_Rate']
    if 'Stance' in df.columns:
        row['Stance'] = df.loc[idx, 'Stance']
    if 'CPI_Inflation' in df.columns:
        row['CPI'] = df.loc[idx, 'CPI_Inflation']
    if 'Nifty50' in df.columns and idx > 0:
        n0 = df.loc[idx-1,'Nifty50']
        n1 = df.loc[idx,  'Nifty50']
        row['Nifty_Return_pct'] = round((n1-n0)/n0*100, 2) if n0 and n0!=0 else np.nan
    ev_rows.append(row)

ev_df = pd.DataFrame(ev_rows)
print("="*70)
print("TABLE 7 — EVENT STUDY RESULTS")
print("="*70)
print(ev_df.to_string(index=False))
ev_df.to_csv(os.path.join(OUTPUT_TAB,'table7_event_study.csv'), index=False)
print("\nSaved: table7_event_study.csv")

TABLE 7 — EVENT STUDY RESULTS
               Event    Date  MPC_Sentiment  Repo_Rate  Rate_Change  Stance  CPI  Nifty_Return_pct
       Taper Tantrum 2013-07        -0.0059       7.25        -0.75 Hawkish  9.8              2.58
Demonetisation Shock 2016-12        -0.0065       6.25        -0.25 Hawkish  3.4              2.04
Rate Cut Cycle Start 2019-06        -0.0076       5.75        -0.75 Hawkish  3.2              8.43
 COVID Emergency Cut 2020-06        -0.0092       4.00        -1.15 Hawkish  6.2            -15.08
Post-COVID Rate Hike 2022-06        -0.0019       4.90         0.90 Neutral  7.3             -3.93
    Rate Cut Restart 2024-12        -0.0044       6.25        -0.25 Hawkish  5.5              1.48

Saved: table7_event_study.csv


In [9]:
print("="*55)
print("NOTEBOOK 05 COMPLETE — ALL TABLES SAVED")
print("="*55)
for fn in ['table1_descriptive_stats.csv','table2_adf_unit_root.csv',
           'table3_correlation_matrix.csv','table4_ols_regression.csv',
           'table5_granger_causality.csv','table7_event_study.csv']:
    exists = os.path.exists(os.path.join(OUTPUT_TAB, fn))
    print("  %s  %s" % ("OK" if exists else "MISSING", fn))
print("\nNext: Run Notebook 06 for all visualizations.")

NOTEBOOK 05 COMPLETE — ALL TABLES SAVED
  OK  table1_descriptive_stats.csv
  OK  table2_adf_unit_root.csv
  OK  table3_correlation_matrix.csv
  OK  table4_ols_regression.csv
  OK  table5_granger_causality.csv
  OK  table7_event_study.csv

Next: Run Notebook 06 for all visualizations.
